<a href="https://colab.research.google.com/github/tizianogallo90/MAIE_MLOps/blob/main/MAIE_TizianoGallo_MLOps_sentiment_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline MLOps per il Monitoraggio del Sentiment sui Social Media

**Destinatario:** Valerio Roncucci — Applied ML Team, MachineInnovators  
**Autore:** Tiziano Gallo  
**Repository GitHub Pubblica:** [https://github.com/tizianogallo90/MAIE_MLOps](https://github.com/tizianogallo90/MAIE_MLOps)

---

## 1. Contesto e Obiettivo del Progetto
Attualmente il monitoraggio della reputazione dei clienti sui social viene svolto manualmente dal team marketing a fine mese, con tre criticità principali:
1. **Latenza decisionale:** eventuali crisi reputazionali emergono con settimane di ritardo.
2. **Soggettività:** valutazioni non uniformi tra revisori diversi.
3. **Scarsa scalabilità:** impossibile processare flussi ad alto volume in tempo reale.

Questo progetto realizza un'infrastruttura **MLOps end-to-end** pronta per la produzione che espone un modello NLP pre-addestrato tramite API REST, garantisce qualità del codice e test automatici in CI/CD, monitora le metriche operative e i segnali di drift in tempo reale e orchestra il ciclo periodico di retraining.

## 2. Architettura e Scelte Tecnologiche

### Struttura della Repository (`MAIE_MLOps`)
```text
MAIE_MLOps/
├── .github/workflows/ci_cd.yml        # Pipeline GitHub Actions (Ruff, Pytest, Docker GHCR, HF Spaces)
├── airflow/dags/
│   └── sentiment_retrain_dag.py       # DAG Airflow settimanale per valutazione drift e retraining
├── app/
│   ├── main.py                        # API FastAPI (/health, /predict, /metrics) con Lifespan
│   ├── model.py                       # Wrapper singleton per RoBERTa sentiment
│   └── schemas.py                     # Validazione contratti input/output con Pydantic
├── monitoring/
│   ├── prometheus/
│   │   ├── prometheus.yml             # Scraping metriche ogni 15s
│   │   └── alert_rules.yml            # Regole di alerting (latenza p95, calo confidenza, errori 5xx)
│   └── grafana/provisioning/          # Datasource e dashboard JSON pre-configurate
├── tests/
│   ├── test_api.py                    # Test funzionali sugli endpoint HTTP
│   └── test_model.py                  # Test comportamentali (direzionalità e invarianza)
├── Dockerfile                         # Build ottimizzata CPU-only con pre-caching dei pesi
├── docker-compose.yml                 # Stack completo: FastAPI + Prometheus + Grafana
├── pyproject.toml                     # Configurazione Ruff (lint/format) e Pytest
└── requirements.txt                   # Dipendenze bloccate per riproducibilità

In [4]:
###  Clone della Repository ed Esecuzione Suite Quality & Pytest

!git clone https://github.com/tizianogallo90/MAIE_MLOps.git
%cd MAIE_MLOps

!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r requirements.txt ruff pytest httpx

print("=== 1. Controllo Linting e Formatting (Ruff) ===")
!ruff check .
!ruff format --check .

print("\n=== 2. Esecuzione Test Unitari e Comportamentali (Pytest) ===")
!pytest -v

Cloning into 'MAIE_MLOps'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 34 (delta 3), reused 33 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 7.16 KiB | 2.39 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/MAIE_MLOps/MAIE_MLOps
=== 1. Controllo Linting e Formatting (Ruff) ===
All checks passed!
7 files already formatted

=== 2. Esecuzione Test Unitari e Comportamentali (Pytest) ===
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/MAIE_MLOps/MAIE_MLOps
configfile: pyproject.toml
testpaths: tests
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 7 items                                                              

tests/test_api.py::test_health_endpoint PASSED   

In [5]:
!pip install -q datasets scikit-learn pandas

import pandas as pd
from datasets import load_dataset
from fastapi.testclient import TestClient
from sklearn.metrics import classification_report, confusion_matrix

from app.main import app

# Caricamento di un campione pubblico con etichette reali di sentiment
dataset = load_dataset("tweet_eval", "sentiment", split="test[:150]")
label_map = {0: "negative", 1: "neutral", 2: "positive"}

y_true = []
y_pred = []
records = []

# Test end-to-end attraverso gli endpoint reali di FastAPI (/health e /predict)
with TestClient(app) as client:
    health = client.get("/health")
    assert health.status_code == 200
    print("Health Check API:", health.json())

    for row in dataset:
        text = row["text"]
        true_label = label_map[row["label"]]
        response = client.post("/predict", json={"text": text})
        assert response.status_code == 200
        pred_data = response.json()

        y_true.append(true_label)
        y_pred.append(pred_data["label"])
        records.append({
            "text": text[:80] + ("..." if len(text) > 80 else ""),
            "true_label": true_label,
            "pred_label": pred_data["label"],
            "confidence": pred_data["score"],
        })

    # Verifica esposizione metriche Prometheus dopo il batch di richieste
    metrics_resp = client.get("/metrics")
    print("Status endpoint /metrics:", metrics_resp.status_code)

print("\n=== Esempi di Predizione ===")
display(pd.DataFrame(records).head(8))

print("\n=== Classification Report sul campione tweet_eval ===")
print(classification_report(y_true, y_pred, digits=3))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

... (2 outputs left out)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Health Check API: {'status': 'ok', 'model': 'cardiffnlp/twitter-roberta-base-sentiment-latest'}
Status endpoint /metrics: 200

=== Esempi di Predizione ===


,text,true_label,pred_label,confidence
0,@user @user what do these '1/2 naked pics' hav...,neutral,negative,0.8047
1,OH: “I had a blue penis while I was this” [pla...,neutral,neutral,0.8669
2,"@user @user That's coming, but I think the vic...",neutral,neutral,0.7637
3,I think I may be finally in with the in crowd ...,positive,positive,0.7740
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative,neutral,0.4164
5,Savchenko now Saakashvili took drug test live ...,neutral,neutral,0.4964
6,How many more days until opening day? 😩,neutral,neutral,0.4666
7,Twitter's #ThankYouObama Shows Heartfelt Grati...,positive,positive,0.9480



=== Classification Report sul campione tweet_eval ===
              precision    recall  f1-score   support

    negative      0.650     0.830     0.729        47
     neutral      0.814     0.658     0.727        73
    positive      0.774     0.800     0.787        30

    accuracy                          0.740       150
   macro avg      0.746     0.762     0.748       150
weighted avg      0.754     0.740     0.740       150



## 3. Pipeline CI/CD (`.github/workflows/ci_cd.yml`)

Ad ogni `push` o `pull_request` sul branch `main`, GitHub Actions esegue tre job concatenati:

1. **`lint-and-test` (CI):** installa le dipendenze (sfruttando la cache dei pesi HuggingFace), verifica stile e formattazione con `ruff check .` e `ruff format --check .`, ed esegue `pytest -v`. Se anche un solo test fallisce, la pipeline si interrompe.
2. **`build-and-push` (CD Registry):** sui push verso `main`, costruisce l'immagine Docker e la pubblica su **GitHub Container Registry (`ghcr.io/tizianogallo90/maie_mlops`)** con tag `latest` e tag immutabile `sha-<commit>`.
3. **`deploy-huggingface` (CD Cloud):** utilizza `huggingface_hub` (`HfApi.upload_folder`) per sincronizzare l'applicazione su un **Docker Space** di HuggingFace:
   * Genera dinamicamente l'intestazione YAML richiesta da Spaces (`sdk: docker`, `app_port: 8000`) in testa al `README.md`.
   * Esclude le cartelle non necessarie al runtime (`airflow/`, `monitoring/`, `tests/`) per velocizzare il build remoto.
   * Si autentica tramite i GitHub Repository Secrets `HF_TOKEN` (token con permesso `write`) e `HF_SPACE_ID` (es. `tizianogallo90/sentiment-api`). Se i secret non sono configurati, lo step esegue uno skip controllato senza bloccare la pipeline.

## 4. Monitoraggio, Soglie di Alerting e Segnali di Drift

### Architettura di Osservabilità (`docker-compose.yml`)
Lo stack avvia tre servizi isolati su rete bridge (`mlops-net`):
* **`sentiment-api` (`:8000`):** espone metriche HTTP standard (RPS, latenza, codici di stato) e due metriche di dominio custom:
  * `sentiment_predictions_total{label}` (Counter per classe predetta)
  * `sentiment_prediction_confidence` (Histogram della confidenza massima)
* **`prometheus` (`:9090`):** raccoglie le metriche ogni 15 secondi e valuta continuamente `alert_rules.yml`.
* **`grafana` (`:3000`):** carica automaticamente via provisioning il datasource Prometheus e la dashboard *Social Sentiment MLOps Monitoring*.

> *(Nota: inserire qui sotto lo screenshot della dashboard Grafana avviata con `docker compose up` trascinando il file `.png` nella cella)*

---

### Metriche Monitorate e Soglie di Alert Configurate

| Alert | Metrica / Espressione PromQL | Soglia e Finestra | Motivazione Operativa |
| :--- | :--- | :--- | :--- |
| **`HighInferenceLatencyP95`** | 95° percentile latenza su `POST /predict` | `> 500 ms` per `2m` | Garantisce reattività quasi real-time sui flussi social ed evidenzia saturazione CPU. |
| **`LowModelConfidenceDriftSignal`** | Media mobile di `sentiment_prediction_confidence` | `< 0.65` su `15m` | Segnale precoce che il modello sta ricevendo testi ambigui, fuori dominio o in una lingua non supportata. |
| **`HighPredictionErrorRate`** | Rate risposte HTTP `5xx` su `/predict` | `> 0.05 req/s` per `1m` | Intercetta immediatamente crash applicativi o errori di memoria del container. |

---

### Come riconoscere Data Drift e Concept Drift sui testi social
In produzione non disponiamo subito delle etichette reali (`ground truth`) per ogni commento in arrivo; per questo motivo monitoriamo indicatori proxy su due livelli:

1. **Data Drift (variazione nella distribuzione degli input $P(X)$):**
   * **Calo dello score di confidenza medio:** quando i testi cambiano registro (es. passaggio da commenti brevi a recensioni tecniche lunghe, o ingresso di commenti in italiano/spagnolo su un modello addestrato prevalentemente su inglese), la distribuzione softmax si appiattisce vicino a `[0.33, 0.33, 0.33]` e l'alert `LowModelConfidenceDriftSignal` scatta.
   * **Aumento dei token troncati / lunghezza testi:** monitoraggio della distribuzione della lunghezza in token dei commenti rispetto alla baseline di validation.
   * **Shift nello spazio degli embedding (batch settimanale):** calcolo del **Population Stability Index (PSI)** sulle distribuzioni di confidenza e della distanza del coseno tra il centroide degli embedding `[CLS]` storici e quelli della finestra corrente (soglia di attenzione: `PSI > 0.20`).

2. **Concept Drift (variazione nella relazione tra testo e sentiment $P(Y|X)$):**
   * **Slang, meme e sarcasmo emergente:** parole storicamente neutre o positive possono assumere valenza fortemente negativa durante un evento specifico (es. il nome di una nuova funzionalità prodotto usato in modo ironico dopo un disservizio).
   * **Anomalie nella distribuzione delle classi predette:** un salto improvviso e non giustificato da campagne marketing (es. classe `neutral` che passa dal 25% al 65%) segnala perdita di capacità discriminante.
   * **Audit Human-in-the-Loop:** il team marketing etichetta un campione stratificato settimanale (es. 500-600 commenti, dando priorità a quelli con confidenza bassa) per misurare il **Macro F1-score** reale e rilevare eventuali cali sotto la soglia di guardia (`0.78`).

## 5. Orchestrazione Continuous Training (Airflow) e Sviluppi Futuri

### Ciclo di Retraining (`airflow/dags/sentiment_retrain_dag.py`)
Il DAG è schedulato settimanalmente (`@weekly`) e implementa una logica **data-driven** tramite TaskFlow API:
1. **`extract_recent_social_data`:** raccoglie i commenti della settimana e il campione validato dal team marketing.
2. **`evaluate_model_and_drift`:** calcola `macro_f1`, `psi_confidence` e `avg_confidence`.
3. **`decide_retraining_branch` (`@task.branch`):** attiva il fine-tuning incrementale **solo se** `Macro F1 < 0.78`, `PSI > 0.20` oppure `avg_confidence < 0.65`; altrimenti salta l'addestramento (`skip_retraining`) risparmiando risorse di calcolo.
4. **`retrain_and_validate_challenger` & `trigger_cd_rollout`:** esegue il fine-tuning di RoBERTa, confronta il nuovo modello (*Challenger*) con quello in produzione (*Champion*) e lo promuove solo se migliora l'F1 di almeno `+0.02`.

---

### Cosa farei diversamente con più tempo
1. **Ottimizzazione delle performance di inferenza (ONNX Runtime + Quantizzazione INT8):** convertire RoBERTa in formato ONNX quantizzato ridurrebbe l'impronta RAM del container da ~1.5 GB a ~450 MB e abbatterebbe la latenza p95 su CPU del 50-60%, consentendo inoltre di introdurre un endpoint batch (`POST /predict/batch`) per elaborare flussi massivi.
2. **Active Learning mirato:** invece di far etichettare al marketing commenti casuali per il audit settimanale, filtrerei automaticamente i messaggi con **entropia predittiva più alta** (confidenza tra `0.45` e `0.60`), massimizzando il guadagno informativo di ogni singola etichetta umana.
3. **Canary Deployment e Shadow Mode:** prima di sostituire il modello in produzione dopo un retraining Airflow, esporrei il *Challenger* in *shadow mode* su una quota del traffico reale per confrontare distribuzione delle predizioni e latenze senza impattare i clienti.
4. **Supporto Multilingua Nativo:** per clienti con community fortemente italiane o multilingua, affiancherei (o sostituirei) `twitter-roberta-base-sentiment-latest` con un modello multilingua come `cardiffnlp/twitter-xlm-roberta-base-sentiment`, aggiungendo un filtro automatico di *language detection* a monte dell'API.